# 用户交互次数分析

分析ml-1m数据集中用户的交互次数分布，为LLM序列化兴趣提取提供依据。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. 加载评分数据

In [ ]:
# 加载RecBole格式的.inter文件
ratings_file = Path('../data/recbole/ml-1m/ml-1m.inter')

if not ratings_file.exists():
    print(f"文件不存在: {ratings_file}")
    print("请先运行 baselines/prepare_data_for_recbole.py")
else:
    df = pd.read_csv(ratings_file, sep='\t')
    # 移除列名的类型后缀
    df.columns = [col.split(':')[0] for col in df.columns]
    print(f"✓ 加载完成")
    print(f"  数据规模: {len(df):,} 条评分")
    print(f"  用户数: {df['user_id'].nunique():,}")
    print(f"  物品数: {df['item_id'].nunique():,}")
    print()
    df.head()

## 2. 用户交互次数分析

In [ ]:
# 按用户统计交互次数
user_interactions = df.groupby('user_id').size().reset_index(name='num_interactions')

print("用户交互次数统计:")
print(user_interactions['num_interactions'].describe())
print()

# 百分位数
percentiles = [10, 25, 50, 75, 90, 95, 99]
print("百分位数:")
for p in percentiles:
    val = np.percentile(user_interactions['num_interactions'], p)
    print(f"  {p}%: {val:.0f}次")

## 3. 可视化分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 直方图
axes[0].hist(user_interactions['num_interactions'], bins=50, edgecolor='black')
axes[0].set_xlabel('交互次数')
axes[0].set_ylabel('用户数')
axes[0].set_title('用户交互次数分布')
axes[0].axvline(user_interactions['num_interactions'].median(), 
                color='red', linestyle='--', label=f'中位数: {user_interactions["num_interactions"].median():.0f}')
axes[0].axvline(user_interactions['num_interactions'].mean(), 
                color='green', linestyle='--', label=f'平均值: {user_interactions["num_interactions"].mean():.0f}')
axes[0].legend()

# 箱线图
axes[1].boxplot(user_interactions['num_interactions'], vert=True)
axes[1].set_ylabel('交互次数')
axes[1].set_title('用户交互次数箱线图')

plt.tight_layout()
plt.show()

## 4. 按时间排序分析（序列化提取需要）

In [ ]:
# 按时间戳排序
df_sorted = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

# 为每个用户的交互添加序号
df_sorted['interaction_seq'] = df_sorted.groupby('user_id').cumcount() + 1

print("时间序列化后的数据示例:")
print(df_sorted[['user_id', 'item_id', 'rating', 'timestamp', 'interaction_seq']].head(20))

## 5. LLM调用成本估算

假设使用序列化LLM提取方法：每K次交互调用一次LLM

In [ ]:
total_interactions = len(df)
num_users = df['user_id'].nunique()
avg_interactions_per_user = total_interactions / num_users

print(f"总交互数: {total_interactions:,}")
print(f"用户数: {num_users:,}")
print(f"平均交互/用户: {avg_interactions_per_user:.1f}")
print()

# 不同batch策略的LLM调用次数
strategies = {
    '每1次交互': 1,
    '每5次交互': 5,
    '每10次交互': 10,
    '每20次交互': 20,
    '每50次交互': 50
}

print("不同策略的LLM调用次数:")
for name, k in strategies.items():
    num_calls = sum(np.ceil(user_interactions['num_interactions'] / k))
    
    # 成本估算（假设gpt-4o-mini，每次调用~$0.001）
    cost_per_call = 0.001  # 假设值
    total_cost = num_calls * cost_per_call
    
    print(f"  {name:15s}: {num_calls:8,.0f} 次调用, 估计成本 ${total_cost:6.2f}")

## 6. 评分分布（用于理解用户偏好）

In [ ]:
print("评分分布:")
print(df['rating'].value_counts().sort_index())
print()

plt.figure(figsize=(10, 5))
df['rating'].value_counts().sort_index().plot(kind='bar')
plt.xlabel('评分')
plt.ylabel('次数')
plt.title('评分分布')
plt.xticks(rotation=0)
plt.show()

# 高分比例（>=4）
high_rating_ratio = (df['rating'] >= 4).mean()
print(f"高分(>=4)占比: {high_rating_ratio:.1%}")

## 7. 活跃时间分桶策略分析（最终方案）

**方案设计：**
- **短期兴趣**：每21天活跃时间为一个桶，统计提取（高分电影知识点）
- **长期兴趣**：每4个短期桶（84天活跃时间），LLM总结一次
- **关键**：按用户实际活跃时间分桶，而非全局日历时间

计算每个用户需要多少次LLM调用。

In [ ]:
# 按用户活跃时间分桶
# 对每个用户，将交互按时间排序后，每21天活跃时间作为一个桶

from datetime import datetime, timedelta

# 配置参数
SHORT_TERM_DAYS = 21  # 短期兴趣桶大小
LONG_TERM_BUCKETS = 4  # 每4个短期桶总结一次长期兴趣

# 计算每个用户的桶数和LLM调用次数
user_llm_calls = []

print("按活跃时间分桶分析...")
print("="*70)

for user_id in df['user_id'].unique():
    user_data = df_sorted[df_sorted['user_id'] == user_id].copy()
    
    # 按时间戳排序（已经排序过，但确保）
    user_data = user_data.sort_values('timestamp')
    
    # 计算活跃桶数
    # 方法：累计交互的时间跨度，每21天算一个活跃桶
    timestamps = user_data['timestamp'].values
    
    if len(timestamps) == 0:
        continue
    
    # 计算第一个和最后一个交互的时间差
    time_span_seconds = timestamps[-1] - timestamps[0]
    time_span_days = time_span_seconds / (24 * 3600)
    
    # 活跃天数估算：假设每次交互代表一天的活跃
    # 更精确的方法：统计有交互的不同日期数
    unique_dates = set()
    for ts in timestamps:
        date = datetime.fromtimestamp(ts).date()
        unique_dates.add(date)
    
    active_days = len(unique_dates)
    
    # 计算短期桶数（每21天活跃时间）
    num_short_term_buckets = max(1, int(np.ceil(active_days / SHORT_TERM_DAYS)))
    
    # 计算LLM调用次数（每4个短期桶调用1次）
    # 即使不足4个桶，也至少调用1次（用户总结）
    num_llm_calls = max(1, int(np.ceil(num_short_term_buckets / LONG_TERM_BUCKETS)))
    
    user_llm_calls.append({
        'user_id': user_id,
        'num_interactions': len(user_data),
        'active_days': active_days,
        'time_span_days': time_span_days,
        'num_short_buckets': num_short_term_buckets,
        'num_llm_calls': num_llm_calls
    })

# 转换为DataFrame便于分析
llm_calls_df = pd.DataFrame(user_llm_calls)

print(f"✓ 分析完成")
print(f"  用户数: {len(llm_calls_df):,}")
print()

# 统计信息
print("活跃天数统计:")
print(llm_calls_df['active_days'].describe())
print()

print("短期桶数统计:")
print(llm_calls_df['num_short_buckets'].describe())
print()

print("LLM调用次数统计（每用户）:")
print(llm_calls_df['num_llm_calls'].describe())
print()

# 总LLM调用次数
total_llm_calls = llm_calls_df['num_llm_calls'].sum()
avg_llm_calls = llm_calls_df['num_llm_calls'].mean()

print("="*70)
print("LLM调用成本估算（最终方案）")
print("="*70)
print(f"总LLM调用次数: {total_llm_calls:,}")
print(f"平均每用户调用: {avg_llm_calls:.1f} 次")
print()

# 成本估算
costs = {
    'gpt-4o-mini ($0.001/call)': total_llm_calls * 0.001,
    'gpt-4o-mini ($0.002/call)': total_llm_calls * 0.002,
    'gpt-4o ($0.01/call)': total_llm_calls * 0.01,
}

for model, cost in costs.items():
    print(f"  {model:30s}: ${cost:7.2f}")

print("="*70)

## 8. 可视化LLM调用分布

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 活跃天数分布
axes[0, 0].hist(llm_calls_df['active_days'], bins=50, edgecolor='black')
axes[0, 0].set_xlabel('活跃天数')
axes[0, 0].set_ylabel('用户数')
axes[0, 0].set_title('用户活跃天数分布')
axes[0, 0].axvline(llm_calls_df['active_days'].median(),
                   color='red', linestyle='--',
                   label=f'中位数: {llm_calls_df["active_days"].median():.0f}天')
axes[0, 0].legend()

# 短期桶数分布
axes[0, 1].hist(llm_calls_df['num_short_buckets'], bins=30, edgecolor='black')
axes[0, 1].set_xlabel('短期桶数（21天/桶）')
axes[0, 1].set_ylabel('用户数')
axes[0, 1].set_title('短期桶数分布')
axes[0, 1].axvline(llm_calls_df['num_short_buckets'].median(),
                   color='red', linestyle='--',
                   label=f'中位数: {llm_calls_df["num_short_buckets"].median():.0f}个')
axes[0, 1].legend()

# LLM调用次数分布
axes[1, 0].hist(llm_calls_df['num_llm_calls'], bins=20, edgecolor='black', color='green')
axes[1, 0].set_xlabel('LLM调用次数（每4桶/次）')
axes[1, 0].set_ylabel('用户数')
axes[1, 0].set_title('每用户LLM调用次数分布')
axes[1, 0].axvline(llm_calls_df['num_llm_calls'].median(),
                   color='red', linestyle='--',
                   label=f'中位数: {llm_calls_df["num_llm_calls"].median():.0f}次')
axes[1, 0].legend()

# 交互次数 vs LLM调用次数
axes[1, 1].scatter(llm_calls_df['num_interactions'],
                   llm_calls_df['num_llm_calls'],
                   alpha=0.5)
axes[1, 1].set_xlabel('交互次数')
axes[1, 1].set_ylabel('LLM调用次数')
axes[1, 1].set_title('交互次数 vs LLM调用次数')

plt.tight_layout()
plt.show()

## 9. 最终方案总结

In [ ]:
# 综合建议
print("="*70)
print("最终方案总结")
print("="*70)
print()
print("📊 数据规模:")
print(f"  - 总用户数: {len(llm_calls_df):,}")
print(f"  - 总交互数: {total_interactions:,}")
print(f"  - 平均交互/用户: {avg_interactions_per_user:.1f}")
print(f"  - 平均活跃天数/用户: {llm_calls_df['active_days'].mean():.0f}")
print(f"  - 中位数活跃天数/用户: {llm_calls_df['active_days'].median():.0f}")
print()
print("🎯 提取策略:")
print(f"  - 短期兴趣: 每 {SHORT_TERM_DAYS} 天活跃时间，统计提取（高分电影，≥4星）")
print(f"  - 短期兴趣数量: 前10个（出现次数>1）")
print(f"  - 长期兴趣: 每 {LONG_TERM_BUCKETS} 个短期桶 ({SHORT_TERM_DAYS * LONG_TERM_BUCKETS} 天)，LLM总结")
print(f"  - 长期兴趣数量: 5个（带年龄记录）")
print(f"  - 按用户实际活跃时间分桶，非全局日历时间")
print()
print("💰 成本估算:")
print(f"  - 总LLM调用: {total_llm_calls:,} 次")
print(f"  - 平均调用/用户: {avg_llm_calls:.1f} 次")
print(f"  - 中位数调用/用户: {llm_calls_df['num_llm_calls'].median():.0f} 次")
print(f"  - 预估成本 (gpt-4o-mini): ${total_llm_calls * 0.001:.2f} - ${total_llm_calls * 0.002:.2f}")
print()
print("✅ 方案优势:")
print("  1. 成本可控 (~$60-120)")
print("  2. LLM用于高层次推理（长期兴趣演化）")
print("  3. 统计方法提取短期兴趣（快速、免费）")
print("  4. 理论依据：21天习惯养成，季度总结")
print("  5. 支持消融实验：长期vs短期、统计vs LLM")
print("  6. 符合论文叙事：LLM as information extractor")
print()
print("📝 最终输出 (用户KG):")
print("  - 5个长期兴趣 (relation: long_term_interest)")
print("  - 10个短期兴趣 (relation: short_term_interest)")
print("  - 长期兴趣带年龄标记（连续保留的天数）")
print()
print("📊 分布特征:")
print(f"  - 短期桶数: 平均{llm_calls_df['num_short_buckets'].mean():.1f}, 中位数{llm_calls_df['num_short_buckets'].median():.0f}")
print(f"  - LLM调用: 平均{avg_llm_calls:.1f}, 中位数{llm_calls_df['num_llm_calls'].median():.0f}")
print("="*70)

In [ ]:
# 综合建议
print("="*70)
print("分析结论")
print("="*70)
print(f"1. 数据规模: {num_users:,}个用户, {total_interactions:,}次交互")
print(f"2. 平均交互数: {avg_interactions_per_user:.1f} 次/用户")
print(f"3. 中位数交互数: {user_interactions['num_interactions'].median():.0f} 次/用户")
print()
print("建议的batch策略:")
print("  - 如果成本敏感: 每20次交互 (~$XX)")
print("  - 如果追求效果: 每5-10次交互 (~$XX)")
print("  - 极端精细化: 每1次交互 (~$XX, 成本较高)")
print("="*70)